# 01 — Data discovery

**Tasks:** P0-T03 (connectivity check). Later sections will be added by M1-T01 (data sources) and M1-T02 (AOI selection).

**Purpose:** prove that our Copernicus Data Space Ecosystem (CDSE) credentials work with the Sentinel Hub APIs before building any ingestion code, and start the quota ledger.

**Inputs:** `config/project.yaml` (endpoints, test area, dates), `.env` (`SH_CLIENT_ID`, `SH_CLIENT_SECRET`; never committed).

**Outputs:** cached Catalog results in `data/raw/catalog/` (local only, with a provenance sidecar), a new row in `data/api_usage_log.csv` after the first real API call.

**Run time:** a few seconds. The first run makes about 2 network requests (one OAuth token, one Catalog page); later runs read the cache and make none.

In [1]:
%load_ext autoreload
%autoreload 2

import logging
import platform
import warnings

# tqdm (used by sentinelhub) warns when ipywidgets is absent; progress bars still work as text.
warnings.filterwarnings("ignore", message="IProgress not found")

import numpy as np
import pandas as pd
import sentinelhub

import desertification
from desertification.config import get_path, load_config, load_env
from desertification.sh_client import (
    cached_catalog_search,
    catalog_request_count,
    missing_credentials,
    summarize_scenes,
)
from desertification.usage import append_usage, latest_balance, make_entry, read_usage_log

# WARNING keeps notebook output free of INFO lines with absolute local paths.
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

cfg = load_config()
SEED = cfg["project"]["seed"]
rng = np.random.default_rng(SEED)

print(f"Python {platform.python_version()} | desertification {desertification.__version__} "
      f"| sentinelhub {sentinelhub.__version__} | pandas {pd.__version__}")

Python 3.12.16 | desertification 0.1.0 | sentinelhub 3.12.1 | pandas 3.1.1


## Connectivity check (P0-T03)

**Why:** every later task depends on authenticated access to Sentinel Hub on CDSE. A Catalog API search is the cheapest possible test: it lists the Sentinel-2 L2A scenes over a small area for one month without downloading or processing any imagery.

**Test area:** an ~11 × 11 km box near the Ferlo (Senegal) candidate AOI, for January 2023 (dry season, so mostly clear skies). It's a connectivity test only; the real study AOIs are chosen in M1-T02. Both come from `config/project.yaml`.

**Quota safety:** API calls only happen when `ALLOW_API_CALLS = True`. Results are cached with a provenance sidecar, so re-running the notebook reads the cache and makes no requests.

In [2]:
# Set to True to allow the one-off Catalog search (about 2 requests). Leave False otherwise:
# with a cache present the notebook still runs fully without contacting the API.
ALLOW_API_CALLS = True

load_env(cfg)  # loads .env into the process environment; values are never printed
missing = missing_credentials()
print("Credentials present:", not missing, "" if not missing else f"(missing: {', '.join(missing)})")

check = cfg["connectivity_check"]
print("Endpoint:", cfg["sentinel_hub"]["base_url"])
print("Test area:", check["name"], check["bbox_wgs84"], "| period:", check["time"]["start"], "to", check["time"]["end"])

Credentials present: True 
Endpoint: https://sh.dataspace.copernicus.eu
Test area: ferlo_test [-14.55, 15.45, -14.45, 15.55] | period: 2023-01-01 to 2023-01-31


In [3]:
features, cache_path, called_api = cached_catalog_search(
    cfg,
    name=check["name"],
    bbox_wgs84=check["bbox_wgs84"],
    start=check["time"]["start"],
    end=check["time"]["end"],
    allow_api=ALLOW_API_CALLS and not missing,
)

if features is None:
    print("No cached results and API calls are off (or credentials are missing).")
    print("Set ALLOW_API_CALLS = True with a filled-in .env and re-run to perform the check.")
    scenes = summarize_scenes([])
else:
    source = "API (fresh request)" if called_api else "cache"
    print(f"Loaded {len(features)} catalog features from {source}: {cache_path.relative_to(cfg['_root'])}")
    scenes = summarize_scenes(features)

Loaded 6 catalog features from API (fresh request): data/raw/catalog/ferlo_test_2023-01-01_2023-01-31.json


In [4]:
if scenes.empty:
    print("No scenes to summarise.")
else:
    print(f"Distinct scenes: {len(scenes)}")
    print(f"Distinct acquisition dates: {scenes['date'].nunique()}")
    print(f"MGRS tiles: {', '.join(sorted(scenes['tile_id'].dropna().unique()))}")
    print("Cloud cover (%):", scenes["cloud_cover_pct"].describe()[["min", "50%", "max"]].round(1).to_dict())
    print(f"Scenes with cloud cover < 10%: {(scenes['cloud_cover_pct'] < 10).sum()}")
scenes

Distinct scenes: 6
Distinct acquisition dates: 6
MGRS tiles: 28PEC
Cloud cover (%): {'min': 0.0, '50%': 37.2, 'max': 98.7}
Scenes with cloud cover < 10%: 2


,id,datetime,date,tile_id,platform,cloud_cover_pct
0,S2A_MSIL2A_20230104T112451_N0510_R037_T28PEC_2...,2023-01-04 11:37:05.735+00:00,2023-01-04,28PEC,sentinel-2a,68.34
1,S2B_MSIL2A_20230109T112339_N0510_R037_T28PEC_2...,2023-01-09 11:37:03.367+00:00,2023-01-09,28PEC,sentinel-2b,98.68
2,S2A_MSIL2A_20230114T112421_N0510_R037_T28PEC_2...,2023-01-14 11:37:03.798+00:00,2023-01-14,28PEC,sentinel-2a,22.03
3,S2B_MSIL2A_20230119T112309_N0510_R037_T28PEC_2...,2023-01-19 11:37:04.250+00:00,2023-01-19,28PEC,sentinel-2b,52.34
4,S2A_MSIL2A_20230124T112341_N0510_R037_T28PEC_2...,2023-01-24 11:37:03.362+00:00,2023-01-24,28PEC,sentinel-2a,0.00
5,S2B_MSIL2A_20230129T112229_N0510_R037_T28PEC_2...,2023-01-29 11:37:04.254+00:00,2023-01-29,28PEC,sentinel-2b,5.41


### Quota ledger

The ledger `data/api_usage_log.csv` starts with the account's balance (30,000 PU / 30,000 requests). After a fresh API call, the cell below appends a row with the number of requests made. The PU and request balances can't be read through the API, so copy them from the [CDSE Sentinel Hub dashboard](https://shapps.dataspace.copernicus.eu/dashboard/) into `PU_BEFORE`/`PU_AFTER`/`REQUESTS_BEFORE`/`REQUESTS_AFTER` before running it (leave as `None` if unknown and fill the CSV in later).

In [6]:
PU_BEFORE = 30000
PU_AFTER = 30000
REQUESTS_BEFORE = 30000
REQUESTS_AFTER = 30000

log_path = get_path(cfg, "api_usage_log")
if called_api:
    # +1 for the OAuth token request made by sentinelhub before the Catalog search.
    n_requests = catalog_request_count(len(features)) + 1
    append_usage(log_path, make_entry(
        task_id="P0-T03", api="catalog", budget_category="setup_pilot",
        request_count=n_requests,
        purpose=f"Connectivity check: Catalog search {check['name']} {check['time']['start']}..{check['time']['end']}",
        pu_before=PU_BEFORE, pu_after=PU_AFTER,
        requests_before=REQUESTS_BEFORE, requests_after=REQUESTS_AFTER,
    ))
    print(f"Logged {n_requests} request(s) in {log_path.relative_to(cfg['_root'])}")
else:
    print("No API call in this run, so nothing new to log.")

usage = read_usage_log(log_path)
print("Latest recorded balance:", latest_balance(usage))
pd.DataFrame(usage)

Logged 2 request(s) in data/api_usage_log.csv
Latest recorded balance: {'pu': 30000.0, 'requests': 30000.0}


,date,task_id,api,budget_category,request_count,pu_before,pu_after,requests_before,requests_after,purpose
0,2026-11-20,P0-T03,balance,setup_pilot,0.0,30000.0,30000.0,30000.0,30000.0,Starting balance on the CDSE account
1,2026-11-20,P0-T03,catalog,setup_pilot,2.0,30000.0,NaN,30000.0,NaN,Connectivity check: Catalog search ferlo_test ...
2,2026-11-20,P0-T03,catalog,setup_pilot,2.0,30000.0,30000.0,30000.0,30000.0,Connectivity check: Catalog search ferlo_test ...


### What to check

- **Acceptance:** the search returns at least one scene (Sentinel-2 revisits every ~5 days, so expect several scenes in a month).
- **No secrets:** this notebook prints only whether credentials are present. Before committing, run `python scripts/check_secrets.py`.
- **Quota:** compare the dashboard balances before and after; the difference tells us whether Catalog requests consume processing units (open question in ROADMAP §12).